---
execute:
    echo: false
---

# Elite Dangerous Database Reader
> Read Elite Dangerous database files

In [ ]:
#| default_exp eddbreader

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

import sys, time, logging, json, gzip

from pathlib import Path
from edcompanion.core import init_configuration

In [ ]:
#| exporti
syslog = logging.getLogger(f"root.{__name__}")


In [ ]:
#| export
def dbfilereader(filename):
    """
        Opens 'filename' as generator for eddb style objects
    """

    chunksize = 64 * 1024 * 1024

    with gzip.open(filename, 'rt') as jsonfile:

        while True:
            chunk = jsonfile.readlines(chunksize)
            if chunk:
                for line in chunk:
                    if len(line) < 5:
                        continue

                    yield json.loads(line.rstrip(',\n\r '))

            else:
                break




In [ ]:
#| export

async def dbfile_process_async(filename, process_chunk):
    """Opens file and calls process_chunk to process batches of items"""

    chunksize = 16 * 1024 * 1024

    with gzip.open(filename, 'rt') as jsonfile:

        while True:
            chunk = jsonfile.readlines(chunksize)
            if chunk:
                data = []
                for line in chunk:
                    if len(line) < 5:
                        continue

                    item = json.loads(line[0:-2]) if line[-2] == "," else json.loads(line)
                    data.append(item)

                await process_chunk(data)

            else:
                break



In [ ]:
#| hidey
import nbdev; nbdev.nbdev_export()